In [ ]:
from utils.parser import benchmarking, benchmarking_batch, compute_acc
from utils.dataloader import load_data, load_data_batch
import json

if __name__ == "__main__":
    '''
    load data in batch from the data folder by fields
    '''
    # field = "general"
    # data = load_data_batch(field=field)
    
    '''
    load test example for testing
    '''
    data_path = "data/tmp.json"
    image_path = "data/tmp"
    data = load_data(data_path, image_path)

    '''
    benchmarking the model using one question
    '''
    field = "general"
    # available models:
    # llama2_7b, llama2_13b, llama3_8b_instruct, mistrial_7b, gpt35, gpt4o, gpt4
    return_data = benchmarking(data, field, model_name='gpt4o')
    results, comparisons = compute_acc(return_data)

    print(f"results: {results}")
    print(json.dumps({"comparison": comparisons}, ensure_ascii=False, indent=4))
    '''
    benchmarking the model using batch of questions
    '''
    # benchmarking_batch(data)


In [ ]:
import os
from PIL import Image

# 获取图像文件夹路径
image_folder = "data/tmp"

# 列出文件夹中的所有文件
image_files = os.listdir(image_folder)

# 遍历每个文件并重命名和压缩
for index, image_file in enumerate(image_files):
    # 获取文件的完整路径
    old_file_path = os.path.join(image_folder, image_file)
    
    # 打开图像并压缩
    with Image.open(old_file_path) as img:
        img = img.convert("RGB")  # 确保图像是RGB格式
        img = img.resize((img.width // 2, img.height // 2))  # 将图像尺寸缩小一半
        
        # 构建新的文件名和路径
        new_file_name = f"image_{index}.png"
        new_file_path = os.path.join(image_folder, new_file_name)
        
        # 保存压缩后的图像
        img.save(new_file_path, "PNG")

print("Images have been renamed and compressed based on their index.")


In [ ]:
from openai import OpenAI

api_key = "4ik5Y3F4-TbXAzhu7Q12uIeg-U9AGYGUQqMlsMZdjK4"
base_url = "http://127.0.0.1:10000/v1/"

def query_gpt35(prompt, images=None, image_captions=None):
    openai_api_key = api_key
    openai_api_base = base_url
    client = OpenAI(api_key=openai_api_key, base_url=openai_api_base)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    llm_response = client.chat.completions.create(
        messages=messages,
        model="GPT-3.5-Turbo",
        max_tokens=1024,
        temperature=0.7,
        stream=False  
    )
    llm_outputs = llm_response.choices[0].message.content
    return llm_outputs

prompt = "hello"

response = query_gpt35(prompt)
print(response)

### Count the types of questions

In [ ]:
import json

data_path = "/home/xiangyu/project/multimodalEDABenchmarking/data/backend.json"
# data_path = "/home/xiangyu/project/multimodalEDABenchmarking/data/rtl.json"


with open(data_path, 'r') as file:
    data = json.load(file)

    types = []
    for key in data.keys():
        instance_types = data[key]['question_type']
        for i in range(len(instance_types)):
            if instance_types[i] not in types:
                types.append(instance_types[i])


    print(types)


In [ ]:
from PIL import Image
from cv2 import resize

image_path = "/home/xiangyu/project/multimodalEDABenchmarking/data/spec/2N2222_page2.png"
image = Image.open(image_path)
# image.show()

image = image.resize((1024, 1024))
# image.show()

# 获取图像的宽度和高度
width, height = image.size
print(width, height)
# 计算新的尺寸，保持长边为512
if width > height:
    new_width = 1024
    new_height = int((1024 / width) * height)
else:
    new_height = 1024
    new_width = int((1024 / height) * width)

# 调整图像大小
resized_image = image.resize((new_width, new_height))

resized_image.show()

### Remove indent from jsonlines

In [5]:
import json
import jsonlines
import os

def read_multiline_json(file_path):
    data = {}
    with open(file_path, 'r', encoding='utf-8') as file:
        buffer = ""
        for line in file:
            line = line.strip()
            if line:  # 忽略空行
                buffer += line
                try:
                    # 尝试解析 JSON 对象
                    json_object = json.loads(buffer)
                    # print(json_object)
                    if list(json_object.values())[0] == "None":
                        buffer = ""
                        continue
                    data[list(json_object.keys())[0]] = list(json_object.values())[0]
                    buffer = ""  # 清空缓冲区以准备下一个 JSON 对象
                except json.JSONDecodeError:
                    # 如果解析失败，继续累积行
                    continue
    return data

base_path = "/home/xiangyu/project/multimodalEDABenchmarking/results/predictedResults/"
field = "rtl/"

file_list_path = base_path + field

file_list = []
for root, dirs, files in os.walk(file_list_path):
    for file in files:
        # file_list.append(os.path.join(root, file))
        print(file)

        file_name_split = file.split('.')
        file_name = file_name_split[0]
        suffix = ".jsonl"
        file_path = base_path + field + file_name + suffix

        data = read_multiline_json(file_path)

        tmp_base_path = "/home/xiangyu/project/multimodalEDABenchmarking/results/tmppredictedResults/"
        tmp_base_path = tmp_base_path + field
        if not os.path.exists(tmp_base_path):
            os.makedirs(tmp_base_path)
        save_path = tmp_base_path + file_name + suffix
        with open(save_path, 'a', encoding='utf-8') as f:
            # Remove indent to ensure we can directly load the file using jsonlines
            for key, returned_response in data.items():
                json.dump({key: returned_response}, f, ensure_ascii=False)
                f.write('\n')

rtl_qwen_2_5_72b_instruct_20241116_090201.jsonl
rtl_qwen_2_0_5b_instruct_20241115_092856.jsonl
rtl_gpt35_20241114_215750.jsonl
rtl_minigpt4_vicuna7b_20241116_090121.jsonl
rtl_minicpm_v2_20241115_173454.jsonl
rtl_llama3_1_8b_instruct_20241115_114207.jsonl
rtl_instructblip_flan_t5_xxl_20241115_101408.jsonl
rtl_internvl2_8b_20241115_173340.jsonl
rtl_yi_vl_6b_20241114_222229.jsonl
